In [ ]:
%load_ext cash
%cash_on
%cash_badge print
%cash_debug on

# Project 2: Wikipedia Pageview Trend Analysis (June 2024)

**Goal**: Analyze Wikipedia hourly pageview data to identify trending topics, time-of-day patterns, cross-language popularity differences, and the "Euro 2024 effect" (tournament started June 14).

**Data**: Wikimedia hourly pageview dumps (~40-70 MB each, gzipped TSV)  
**Source**: https://dumps.wikimedia.org/other/pageviews/2024/2024-06/

**Cash Stress Points**: 
- Multiple file ingestion (24 files per day × 7 days = 168 files)
- Large string-heavy DataFrames (article titles)
- Chunked concat operations
- Rolling window time series analysis

In [ ]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import gzip
import os
import time
import urllib.request
from datetime import datetime

In [ ]:
# Download Wikipedia pageview data - using simple index loop 
# (tuple unpacking in for-loops has a Cash control_structures bug)
data_dir = os.path.join(os.getcwd(), 'examples', 'large_scale_projects', 'data', 'wikipedia')
os.makedirs(data_dir, exist_ok=True)

base_url = 'https://dumps.wikimedia.org/other/pageviews/2024/2024-06/'
dates = ['20240601', '20240602', '20240603']  # 3 days = 72 files
hours = [f'{h:02d}0000' for h in range(24)]

# Build flat lists (avoid tuple unpacking in loop)
urls = []
fpaths = []
for date in dates:
    for hour in hours:
        fname = f'pageviews-{date}-{hour}.gz'
        fpath = os.path.join(data_dir, fname)
        if not os.path.exists(fpath):
            urls.append(f'{base_url}{fname}')
            fpaths.append(fpath)

print(f"Total hourly files needed: {len(dates) * len(hours)}")
print(f"Already downloaded: {len(dates) * len(hours) - len(urls)}")
print(f"To download: {len(urls)}")

t0 = time.time()
failed = 0
for i in range(len(urls)):
    try:
        urllib.request.urlretrieve(urls[i], fpaths[i])
        if (i + 1) % 12 == 0:
            elapsed = time.time() - t0
            print(f"  Downloaded {i+1}/{len(urls)} ({elapsed:.0f}s elapsed)")
    except Exception as e:
        failed += 1
        print(f"  FAILED #{i}: {e}")

elapsed = time.time() - t0
print(f"\nDownload complete: {len(urls) - failed}/{len(urls)} files in {elapsed:.0f}s")

# Report total size
total_size = sum(os.path.getsize(os.path.join(data_dir, f)) 
                 for f in os.listdir(data_dir) if f.startswith('pageviews'))
print(f"Total data size: {total_size / 1e9:.2f} GB ({total_size / 1e6:.0f} MB)")

In [ ]:
# Parse Wikipedia pageview files - English Wikipedia only
# Each line: domain_code page_title view_count response_size
t0 = time.time()

# Find the actual data directory
possible_dirs = [
    os.path.join(os.getcwd(), 'examples', 'large_scale_projects', 'data', 'wikipedia'),
    os.path.join(os.getcwd(), 'data', 'wikipedia'),
]
for d in possible_dirs:
    if os.path.exists(d) and len(os.listdir(d)) > 0:
        data_dir = d
        break

gz_files = sorted([f for f in os.listdir(data_dir) if f.endswith('.gz')])
print(f"Found {len(gz_files)} pageview files in {data_dir}")

# Parse files - only keep English Wikipedia (domain_code == 'en')  
rows = []
for idx in range(len(gz_files)):
    fpath = os.path.join(data_dir, gz_files[idx])
    # Extract datetime from filename: pageviews-YYYYMMDD-HHMMSS.gz
    parts = gz_files[idx].replace('.gz', '').split('-')
    file_dt = datetime.strptime(f"{parts[1]}-{parts[2]}", "%Y%m%d-%H%M%S")
    
    with gzip.open(fpath, 'rt', encoding='utf-8', errors='replace') as f:
        for line in f:
            fields = line.strip().split(' ')
            if len(fields) >= 3 and fields[0] == 'en':
                try:
                    views = int(fields[2])
                    if views >= 10:  # Filter noise - only pages with 10+ views/hour
                        rows.append((file_dt, fields[1], views))
                except ValueError:
                    pass
    
    if (idx + 1) % 12 == 0:
        elapsed = time.time() - t0
        print(f"  Parsed {idx+1}/{len(gz_files)} files ({elapsed:.0f}s, {len(rows):,} rows so far)")

elapsed = time.time() - t0
print(f"\nParsing complete: {len(rows):,} rows in {elapsed:.0f}s")
print(f"Creating DataFrame...")

df = pd.DataFrame(rows, columns=['timestamp', 'page', 'views'])
df['timestamp'] = pd.to_datetime(df['timestamp'])
print(f"DataFrame: {len(df):,} rows, {df.memory_usage(deep=True).sum() / 1e6:.0f} MB")
print(f"\nDate range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Unique pages: {df['page'].nunique():,}")

In [ ]:
# Top 20 most viewed Wikipedia pages (June 1-3, 2024)
print(f"DataFrame shape: {df.shape}")
print(f"Total rows: {len(df):,}")

top_pages = df.groupby('page')['views'].sum().sort_values(ascending=False).head(20)
print("\nTop 20 most viewed English Wikipedia pages:")
for rank, (page, views) in enumerate(top_pages.items(), 1):
    print(f"  {rank:2d}. {str(page):<40s} {int(views):>10,} views")

total_views = int(df['views'].sum())
print(f"\nTotal tracked views: {total_views:,}")
print(f"Top 20 account for: {top_pages.sum() / total_views * 100:.1f}% of tracked views")

In [ ]:
# === Hourly Traffic Patterns ===
# Self-contained: rebuild df to avoid upstream lineage mismatch (Issue 15)
import pandas as _pd_h
import gzip as _gz_h
import os as _os_h
from datetime import datetime as _dt_h

_wiki_dir = None
for _d in [_os_h.path.join(_os_h.getcwd(), 'examples', 'large_scale_projects', 'data', 'wikipedia'),
           _os_h.path.join(_os_h.getcwd(), 'examples', 'large_scale_projects', 'examples', 'large_scale_projects', 'data', 'wikipedia')]:
    if _os_h.path.isdir(_d):
        _wiki_dir = _d

_gz_h_files = sorted([_f for _f in _os_h.listdir(_wiki_dir) if _f.endswith('.gz')])
print(f"Rebuilding df from {len(_gz_h_files)} files for hourly analysis...")
_rows_h = []
for _i_h, _gzf_h in enumerate(_gz_h_files):
    _fp_h = _os_h.path.join(_wiki_dir, _gzf_h)
    _parts_h = _gzf_h.replace('pageviews-', '').replace('.gz', '').split('-')
    _fdt_h = _dt_h.strptime(f"{_parts_h[0]}-{_parts_h[1]}", "%Y%m%d-%H%M%S")
    with _gz_h.open(_fp_h, 'rt', encoding='utf-8', errors='replace') as _fobj_h:
        for _ln_h in _fobj_h:
            _flds_h = _ln_h.strip().split(' ')
            if len(_flds_h) >= 3 and _flds_h[0] == 'en':
                try:
                    _rows_h.append((_fdt_h, _flds_h[1], int(_flds_h[2])))
                except ValueError:
                    pass

_df_h = _pd_h.DataFrame(_rows_h, columns=['timestamp', 'page', 'views'])
_df_h['hour'] = _df_h['timestamp'].dt.hour
print(f"Built DataFrame: {len(_df_h):,} rows")

_hourly_agg = _df_h.groupby('hour')['views'].agg(
    total_views='sum', mean_views='mean', page_count='count'
)
_pk_hour = int(_hourly_agg['total_views'].idxmax())
_tr_hour = int(_hourly_agg['total_views'].idxmin())
_pk_views = int(_hourly_agg.loc[_pk_hour, 'total_views'])
_tr_views = int(_hourly_agg.loc[_tr_hour, 'total_views'])

print(f"\n=== Hourly Wikipedia Traffic Patterns (June 1-3, 2024) ===\n")
print(_hourly_agg.to_string())
print(f"\n📈 Peak hour:   {_pk_hour}:00 UTC ({_pk_views:,} views)")
print(f"📉 Trough hour: {_tr_hour}:00 UTC ({_tr_views:,} views)")
print(f"📊 Peak/Trough ratio: {_pk_views / _tr_views:.2f}x")

In [ ]:
# === Daily Trends & Top Pages per Day ===
# Reuse _df_h from cell above (67M rows with hour column)
_df_h['date'] = _df_h['timestamp'].dt.date

# Daily totals
_daily = _df_h.groupby('date')['views'].agg(
    total_views='sum', unique_pages='nunique', avg_views='mean'
)
print("=== Daily Wikipedia Pageview Summary (en) ===\n")
print(_daily.to_string())
print(f"\nTotal across all days: {int(_daily['total_views'].sum()):,} views")

# Top 5 pages per day
print("\n=== Top 5 Pages per Day ===")
for _date in sorted(_df_h['date'].unique()):
    _day_df = _df_h[_df_h['date'] == _date]
    _top5 = _day_df.groupby('page')['views'].sum().nlargest(5)
    print(f"\n📅 {_date}:")
    for _rank, (_pg, _vw) in enumerate(_top5.items(), 1):
        print(f"  {_rank}. {str(_pg)[:50]:50s} {int(_vw):>12,} views")

In [ ]:
# === Visualization: Hourly Traffic Pattern ===
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as _plt_v

_fig, (_ax1, _ax2) = _plt_v.subplots(1, 2, figsize=(16, 6))

# Plot 1: Hourly traffic pattern
_hours_list = list(range(24))
_views_list = [int(_hourly_agg.loc[_hh, 'total_views']) for _hh in _hours_list]
_ax1.bar(_hours_list, _views_list, color='steelblue', alpha=0.8)
_ax1.set_xlabel('Hour (UTC)')
_ax1.set_ylabel('Total Views')
_ax1.set_title('Wikipedia Hourly Traffic (en, June 1-3 2024)')
_ax1.set_xticks(range(0, 24, 2))
_ax1.axhline(y=sum(_views_list)/24, color='red', linestyle='--', alpha=0.5, label='Average')
_ax1.legend()

# Plot 2: Daily totals bar chart  
_day_labels = [str(_d) for _d in sorted(_daily.index)]
_day_views = [int(_daily.loc[_d, 'total_views']) for _d in sorted(_daily.index)]
_colors = ['#2196F3', '#4CAF50', '#FF9800']
_ax2.bar(_day_labels, _day_views, color=_colors, alpha=0.8)
_ax2.set_xlabel('Date')
_ax2.set_ylabel('Total Views')
_ax2.set_title('Daily Wikipedia Pageviews (en)')
for _i_v, _v in enumerate(_day_views):
    _ax2.text(_i_v, _v + 500000, f'{_v:,}', ha='center', fontsize=9)

_plt_v.tight_layout()
_plt_v.savefig('examples/large_scale_projects/wikipedia_traffic.png', dpi=150, bbox_inches='tight')
print("📊 Chart saved to wikipedia_traffic.png")
_plt_v.close()